In [9]:
!pip install mediapipe opencv-python pandas scikit-learn

In [10]:
!pip install mediapipe

In [11]:
import mediapipe as mp
import cv2

In [12]:
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [13]:
pose_model = "/Users/anurag_77y/anaconda_projects/pose_landmarker_full.task"
hand_model="/Users/anurag_77y/anaconda_projects/hand_landmarker.task"
face_model="/Users/anurag_77y/anaconda_projects/face_landmarker.task"

In [14]:
BaseOptions=python.BaseOptions

In [15]:
pose_options = vision.PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=pose_model),
    running_mode=vision.RunningMode.VIDEO
)
pose_detector = vision.PoseLandmarker.create_from_options(pose_options)


I0000 00:00:1776459502.249813  420765 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2
W0000 00:00:1776459502.297264  420766 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776459502.306645  420773 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [16]:
hand_options = vision.HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=hand_model),
    running_mode=vision.RunningMode.VIDEO,
    num_hands=2
)
hand_detector = vision.HandLandmarker.create_from_options(hand_options)

I0000 00:00:1776459502.319518  420777 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2
W0000 00:00:1776459502.323500  420779 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776459502.327187  420785 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [17]:
face_options = vision.FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=face_model),
    running_mode=vision.RunningMode.VIDEO
)
face_detector = vision.FaceLandmarker.create_from_options(face_options)

W0000 00:00:1776459502.333151  420787 face_landmarker_graph.cc:180] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
I0000 00:00:1776459502.335729  420787 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2
W0000 00:00:1776459502.336832  420790 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776459502.344345  420789 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [18]:
POSE_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,7),
    (0,4),(4,5),(5,6),(6,8),
    (9,10),
    (11,12),
    (11,13),(13,15),
    (12,14),(14,16),
    (15,17),(16,18),
    (11,23),(12,24),
    (23,24),
    (23,25),(25,27),
    (24,26),(26,28),
    (27,29),(28,30),
    (29,31),(30,32)
]

def draw_pose_skeleton(frame, landmarks):
    h, w, _ = frame.shape

    # Convert all landmarks to pixel coords
    points = []
    for lm in landmarks:
        x = int(lm.x * w)
        y = int(lm.y * h)
        points.append((x, y))

    # Draw lines (bones)
    for connection in POSE_CONNECTIONS:
        start_idx, end_idx = connection
        cv2.line(frame, points[start_idx], points[end_idx], (0,255,0), 2)

    # Draw joints
    for point in points:
        cv2.circle(frame, point, 3, (0,0,255), -1)

In [19]:
cap=cv2.VideoCapture(0)
frame_timestamp=0
while cap.isOpened():
    ret,frame=cap.read()
    if not ret:
        break
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    #run detections
    pose_result=pose_detector.detect_for_video(mp_image,frame_timestamp)
    hand_result=hand_detector.detect_for_video(mp_image,frame_timestamp)
    face_result=face_detector.detect_for_video(mp_image,frame_timestamp)
    frame_timestamp+=1
    if hand_result.hand_landmarks:
        for hand in hand_result.hand_landmarks:
            for landmark in hand:
                x = int(landmark.x * frame.shape[1])
                y = int(landmark.y * frame.shape[0])
                cv2.circle(frame, (x, y), 5, (255,0,0), -1)

    # Face
    if face_result.face_landmarks:
        for face in face_result.face_landmarks:
            for landmark in face:
                x = int(landmark.x * frame.shape[1])
                y = int(landmark.y * frame.shape[0])
                cv2.circle(frame, (x, y), 3, (0,0,255), -1)
    if pose_result.pose_landmarks:
        draw_pose_skeleton(frame, pose_result.pose_landmarks[0])

    # Show
    cv2.imshow("Modern MediaPipe", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()
    


W0000 00:00:1776459503.855961  420781 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


In [20]:
h, w, _ = frame.shape

# Pose
if pose_result.pose_landmarks:
    for i, lm in enumerate(pose_result.pose_landmarks[0]):
        x = int(lm.x * w)
        y = int(lm.y * h)
        print(f"Pose {i}: ({x},{y}), visibility={lm.visibility:.2f}")

# Face
if face_result.face_landmarks:
    for face in face_result.face_landmarks:
        for i, lm in enumerate(face):
            x = int(lm.x * w)
            y = int(lm.y * h)
            print(f"Face {i}: ({x},{y})")

Pose 0: (1062,602), visibility=1.00
Pose 1: (1103,538), visibility=1.00
Pose 2: (1127,539), visibility=1.00
Pose 3: (1151,542), visibility=1.00
Pose 4: (1020,538), visibility=1.00
Pose 5: (992,540), visibility=1.00
Pose 6: (968,543), visibility=1.00
Pose 7: (1190,575), visibility=1.00
Pose 8: (936,580), visibility=1.00
Pose 9: (1116,679), visibility=1.00
Pose 10: (1009,679), visibility=1.00
Pose 11: (1413,915), visibility=1.00
Pose 12: (777,946), visibility=1.00
Pose 13: (1665,1298), visibility=0.22
Pose 14: (519,1322), visibility=0.33
Pose 15: (1540,1661), visibility=0.15
Pose 16: (546,1538), visibility=0.21
Pose 17: (1566,1778), visibility=0.16
Pose 18: (503,1592), visibility=0.23
Pose 19: (1508,1761), visibility=0.18
Pose 20: (572,1552), visibility=0.28
Pose 21: (1470,1721), visibility=0.18
Pose 22: (616,1537), visibility=0.27
Pose 23: (1305,1743), visibility=0.00
Pose 24: (877,1732), visibility=0.00
Pose 25: (1292,2419), visibility=0.00
Pose 26: (859,2407), visibility=0.00
Pose 27:

In [21]:
import numpy as np

In [22]:
import csv
import os

In [23]:
num_pose = len(pose_result.pose_landmarks[0]) if pose_result.pose_landmarks else 33
num_face = len(face_result.face_landmarks[0]) if face_result.face_landmarks else 468


In [24]:
landmarks=['class']
for i in range(1, num_pose + 1):
    landmarks += [f'pose_x{i}', f'pose_y{i}', f'pose_z{i}', f'pose_v{i}']
for i in range(1, num_face + 1):
    landmarks += [f'face_x{i}', f'face_y{i}', f'face_z{i}']

In [25]:
landmarks

['class',
 'pose_x1',
 'pose_y1',
 'pose_z1',
 'pose_v1',
 'pose_x2',
 'pose_y2',
 'pose_z2',
 'pose_v2',
 'pose_x3',
 'pose_y3',
 'pose_z3',
 'pose_v3',
 'pose_x4',
 'pose_y4',
 'pose_z4',
 'pose_v4',
 'pose_x5',
 'pose_y5',
 'pose_z5',
 'pose_v5',
 'pose_x6',
 'pose_y6',
 'pose_z6',
 'pose_v6',
 'pose_x7',
 'pose_y7',
 'pose_z7',
 'pose_v7',
 'pose_x8',
 'pose_y8',
 'pose_z8',
 'pose_v8',
 'pose_x9',
 'pose_y9',
 'pose_z9',
 'pose_v9',
 'pose_x10',
 'pose_y10',
 'pose_z10',
 'pose_v10',
 'pose_x11',
 'pose_y11',
 'pose_z11',
 'pose_v11',
 'pose_x12',
 'pose_y12',
 'pose_z12',
 'pose_v12',
 'pose_x13',
 'pose_y13',
 'pose_z13',
 'pose_v13',
 'pose_x14',
 'pose_y14',
 'pose_z14',
 'pose_v14',
 'pose_x15',
 'pose_y15',
 'pose_z15',
 'pose_v15',
 'pose_x16',
 'pose_y16',
 'pose_z16',
 'pose_v16',
 'pose_x17',
 'pose_y17',
 'pose_z17',
 'pose_v17',
 'pose_x18',
 'pose_y18',
 'pose_z18',
 'pose_v18',
 'pose_x19',
 'pose_y19',
 'pose_z19',
 'pose_v19',
 'pose_x20',
 'pose_y20',
 'pose_z20',

In [26]:
import csv
import os
with open('coords.csv', mode='w', newline='') as f:
    csv_writer = csv.writer(f)
    csv_writer.writerow(landmarks)

In [27]:
cap = cv2.VideoCapture(0)
timestamp = 0

label = None

key_map = {
    ord('h'): 'happy',
    ord('s'): 'sad',
    ord('v'): 'victory',
    ord('n'): 'neutral'
}
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

    # Detection
    pose_result = pose_detector.detect_for_video(mp_image, timestamp)
    face_result = face_detector.detect_for_video(mp_image, timestamp)
    timestamp += 1
    key = cv2.waitKey(1) & 0xFF

    if key in key_map:
        label = key_map[key]
        print(f"Recording: {label}")

    if key == ord('q'):
        break
    
    if label is not None:
        row = [label]

        # Pose
        if pose_result.pose_landmarks:
            for lm in pose_result.pose_landmarks[0]:
                row += [lm.x, lm.y, lm.z, lm.visibility]
        else:
            row += [0] * (33 * 4)

        # Face
        if face_result.face_landmarks:
            for lm in face_result.face_landmarks[0]:
                row += [lm.x, lm.y, lm.z]
        else:
            row += [0] * (468 * 3)

        # Save row
        with open('coords.csv', mode='a', newline='') as f:
            csv_writer = csv.writer(f)
            csv_writer.writerow(row)

        # Optional: small delay to avoid duplicate frames
        cv2.putText(frame, f"Recording: {label}", (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

    # ------------------ DISPLAY ------------------

    cv2.imshow("Data Collection", frame)

cap.release()
cv2.destroyAllWindows()
    

ValueError: Input timestamp must be monotonically increasing.